In [1]:
# ============================================================
# 05_feature_engineering.ipynb — FEATURE ENGINEERING
# Source : data/processed/ → data/processed/olist_features.csv
# Cours activé : Cours 3 §3 — tendance centrale par client
# Cours activé : Cours 1 P4 — One-Hot Encoding
# Cours activé : Cours 2 P2A — features pour ML supervisé
# ============================================================

import sys
sys.path.append('..')
from src.utils import *

# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 1 — CHARGEMENT DONNÉES NETTOYÉES")
print("="*60 + "\n")
# -------------------------------------------------------

orders    = pd.read_csv('../data/processed/orders_clean.csv',
                        parse_dates=['order_purchase_timestamp',
                                     'order_approved_at',
                                     'order_delivered_carrier_date',
                                     'order_delivered_customer_date',
                                     'order_estimated_delivery_date'])
items     = pd.read_csv('../data/processed/items_clean.csv')
payments  = pd.read_csv('../data/processed/payments_clean.csv')
reviews   = pd.read_csv('../data/processed/reviews_clean.csv')
products  = pd.read_csv('../data/processed/products_clean.csv')
customers = pd.read_csv('../data/processed/customers_clean.csv')

# Date de référence pour le calcul de Recency
# On prend le lendemain de la dernière commande dans le dataset
date_ref = orders['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print(f"✓ Date de référence RFM : {date_ref.date()}")
print(f"✓ Tables chargées")


   ÉTAPE 1 — CHARGEMENT DONNÉES NETTOYÉES

✓ Date de référence RFM : 2018-10-18
✓ Tables chargées


In [ ]:
# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 2 — VARIABLES TEMPORELLES")
print("   EDA H1 : délai corrèle avec satisfaction")
print("="*60 + "\n")
# -------------------------------------------------------

# Travailler uniquement sur les commandes livrées
# Les autres statuts n'ont pas de date de livraison complète
orders_livrees = orders[orders['order_status'] == 'delivered'].copy()

# Délai de livraison réel (jours)
orders_livrees['delai_livraison'] = (
    orders_livrees['order_delivered_customer_date'] -
    orders_livrees['order_purchase_timestamp']
).dt.days

# Délai d'approbation (heures)
orders_livrees['delai_approbation'] = (
    orders_livrees['order_approved_at'] -
    orders_livrees['order_purchase_timestamp']
).dt.total_seconds() / 3600

# Retard vs estimation (positif = en retard, négatif = en avance)
orders_livrees['retard_vs_estimation'] = (
    orders_livrees['order_delivered_customer_date'] -
    orders_livrees['order_estimated_delivery_date']
).dt.days

# Variables calendaires — pour la saisonnalité
orders_livrees['mois']       = orders_livrees['order_purchase_timestamp'].dt.month
orders_livrees['trimestre']  = orders_livrees['order_purchase_timestamp'].dt.quarter
orders_livrees['jour_semaine']= orders_livrees['order_purchase_timestamp'].dt.dayofweek
orders_livrees['is_weekend'] = orders_livrees['jour_semaine'].isin([5,6]).astype(int)

print(f"✓ Commandes livrées : {len(orders_livrees)}")
print(f"\n--- Statistiques délais ---")
print(f"  Délai moyen     : {orders_livrees['delai_livraison'].mean():.1f} jours")
print(f"  Délai médian    : {orders_livrees['delai_livraison'].median():.1f} jours")
print(f"  Retard moyen    : {orders_livrees['retard_vs_estimation'].mean():.1f} jours")
print(f"  % commandes en retard : "
      f"{(orders_livrees['retard_vs_estimation'] > 0).mean()*100:.1f}%")


   ÉTAPE 2 — VARIABLES TEMPORELLES
   Cours 3 §2 — variables continues dérivées des dates
   EDA H1 : délai corrèle avec satisfaction

✓ Commandes livrées : 96470

--- Statistiques délais ---
  Délai moyen     : 12.1 jours
  Délai médian    : 10.0 jours
  Retard moyen    : -11.9 jours
  % commandes en retard : 6.8%


In [3]:
# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 3 — VARIABLES BUSINESS PAR COMMANDE")
print("   Revenue, panier, catégories")
print("="*60 + "\n")
# -------------------------------------------------------

# Revenue par commande = prix + fret
items['revenue'] = items['price'] + items['freight_value']

# Agrégation par commande
items_agg = items.groupby('order_id').agg(
    revenue_commande   = ('revenue', 'sum'),
    prix_commande      = ('price', 'sum'),
    fret_commande      = ('freight_value', 'sum'),
    nb_articles        = ('order_item_id', 'max'),
    nb_produits_uniq   = ('product_id', 'nunique'),
).reset_index()

# Nombre de catégories différentes par commande
cat_par_cmd = (items.merge(
    products[['product_id','product_category_name_english']],
    on='product_id', how='left')
    .groupby('order_id')['product_category_name_english']
    .nunique()
    .reset_index()
    .rename(columns={'product_category_name_english':'nb_categories'}))

items_agg = items_agg.merge(cat_par_cmd, on='order_id', how='left')

print(f"✓ Agrégation commandes : {items_agg.shape}")
print(f"\n--- Stats revenue par commande ---")
print(f"  Revenue moyen   : {items_agg['revenue_commande'].mean():.2f} BRL")
print(f"  Revenue médian  : {items_agg['revenue_commande'].median():.2f} BRL")
print(f"  Articles moyen  : {items_agg['nb_articles'].mean():.2f}")


   ÉTAPE 3 — VARIABLES BUSINESS PAR COMMANDE
   Revenue, panier, catégories

✓ Agrégation commandes : (98666, 7)

--- Stats revenue par commande ---
  Revenue moyen   : 160.58 BRL
  Revenue médian  : 105.29 BRL
  Articles moyen  : 1.14


In [4]:
# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 4 — VARIABLES PAIEMENT PAR COMMANDE")
print("   Cours 3 §2 — payment_type nominale → encoding")
print("="*60 + "\n")
# -------------------------------------------------------

# Agrégation paiements par commande
pay_agg = payments.groupby('order_id').agg(
    montant_total_paye  = ('payment_value', 'sum'),
    nb_paiements        = ('payment_sequential', 'max'),
    nb_mensualites_max  = ('payment_installments', 'max'),
).reset_index()

# Type de paiement dominant par commande
pay_type = (payments.groupby('order_id')['payment_type']
            .agg(lambda x: x.value_counts().index[0])
            .reset_index()
            .rename(columns={'payment_type':'payment_type_dominant'}))

pay_agg = pay_agg.merge(pay_type, on='order_id', how='left')

# Flag carte de crédit
pay_agg['is_credit_card'] = (
    pay_agg['payment_type_dominant'] == 'credit_card'
).astype(int)

print(f"✓ Agrégation paiements : {pay_agg.shape}")
print(f"\n--- Stats paiements ---")
print(f"  % carte crédit  : "
      f"{pay_agg['is_credit_card'].mean()*100:.1f}%")
print(f"  Mensualités moy : "
      f"{pay_agg['nb_mensualites_max'].mean():.1f}")


   ÉTAPE 4 — VARIABLES PAIEMENT PAR COMMANDE
   Cours 3 §2 — payment_type nominale → encoding

✓ Agrégation paiements : (99440, 6)

--- Stats paiements ---
  % carte crédit  : 75.7%
  Mensualités moy : 2.9


In [5]:
# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 5 — VARIABLES SATISFACTION PAR COMMANDE")
print("   EDA H2 : satisfaction liée aux catégories et délais")
print("="*60 + "\n")
# -------------------------------------------------------

# Score de satisfaction par commande
reviews_agg = reviews.groupby('order_id').agg(
    score_moyen     = ('review_score', 'mean'),
    nb_avis         = ('review_score', 'count'),
).reset_index()

# Flag insatisfaction
reviews_agg['is_insatisfait'] = (
    reviews_agg['score_moyen'] <= 2
).astype(int)

print(f"✓ Agrégation reviews : {reviews_agg.shape}")
print(f"  % insatisfaits  : "
      f"{reviews_agg['is_insatisfait'].mean()*100:.1f}%")


   ÉTAPE 5 — VARIABLES SATISFACTION PAR COMMANDE
   EDA H2 : satisfaction liée aux catégories et délais

✓ Agrégation reviews : (98673, 4)
  % insatisfaits  : 14.6%


In [6]:
# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 6 — TABLE MAÎTRE PAR COMMANDE")
print("   Assembler toutes les variables par order_id")
print("="*60 + "\n")
# -------------------------------------------------------

# Base : commandes livrées avec customer_unique_id
base = orders_livrees[['order_id','customer_id',
                        'order_purchase_timestamp',
                        'delai_livraison',
                        'delai_approbation',
                        'retard_vs_estimation',
                        'mois','trimestre',
                        'jour_semaine','is_weekend']].copy()

# Ajouter customer_unique_id depuis customers
base = base.merge(
    customers[['customer_id','customer_unique_id','customer_state']],
    on='customer_id', how='left'
)

# Ajouter variables business
base = base.merge(items_agg, on='order_id', how='left')

# Ajouter variables paiement
base = base.merge(pay_agg, on='order_id', how='left')

# Ajouter variables satisfaction
base = base.merge(reviews_agg, on='order_id', how='left')

print(f"✓ Table maître commandes : {base.shape}")
print(f"  Colonnes : {base.columns.tolist()}")
print(f"\n  Nulls par colonne :")
nulls = base.isnull().sum()
print(nulls[nulls > 0])


   ÉTAPE 6 — TABLE MAÎTRE PAR COMMANDE
   Assembler toutes les variables par order_id

✓ Table maître commandes : (96470, 26)
  Colonnes : ['order_id', 'customer_id', 'order_purchase_timestamp', 'delai_livraison', 'delai_approbation', 'retard_vs_estimation', 'mois', 'trimestre', 'jour_semaine', 'is_weekend', 'customer_unique_id', 'customer_state', 'revenue_commande', 'prix_commande', 'fret_commande', 'nb_articles', 'nb_produits_uniq', 'nb_categories', 'montant_total_paye', 'nb_paiements', 'nb_mensualites_max', 'payment_type_dominant', 'is_credit_card', 'score_moyen', 'nb_avis', 'is_insatisfait']

  Nulls par colonne :
delai_approbation         14
montant_total_paye         1
nb_paiements               1
nb_mensualites_max         1
payment_type_dominant      1
is_credit_card             1
score_moyen              646
nb_avis                  646
is_insatisfait           646
dtype: int64


In [ ]:
# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 7 — RFM : AGRÉGATION PAR CLIENT")
print("   EDA : 96.9% mono-achat → Recency critique")
print("="*60 + "\n")
# -------------------------------------------------------

# Agrégation par customer_unique_id
# C'est ici qu'on utilise unique_id — pas customer_id
# Audit étape 10 : 3345 clients ont plusieurs customer_id
rfm = base.groupby('customer_unique_id').agg(
    # RFM classique
    recency         = ('order_purchase_timestamp',
                       lambda x: (date_ref - x.max()).days),
    frequency       = ('order_id', 'count'),
    monetary        = ('revenue_commande', 'sum'),

    # Variables business agrégées
    panier_moyen    = ('prix_commande', 'mean'),
    delai_moyen     = ('delai_livraison', 'mean'),
    retard_moyen    = ('retard_vs_estimation', 'mean'),
    score_moyen     = ('score_moyen', 'mean'),
    nb_avis         = ('nb_avis', 'sum'),
    is_insatisfait  = ('is_insatisfait', 'max'),
    nb_articles_moy = ('nb_articles', 'mean'),
    nb_cat_moy      = ('nb_categories', 'mean'),
    pct_credit      = ('is_credit_card', 'mean'),
    mensualites_moy = ('nb_mensualites_max', 'mean'),

    # Info géographique
    customer_state  = ('customer_state', 'first'),

    # Temporel
    mois_premier_achat = ('order_purchase_timestamp', 'min'),
    mois_dernier_achat = ('order_purchase_timestamp', 'max'),

).reset_index()

print(f"✓ Table RFM : {rfm.shape}")
print(f"\n--- Stats RFM ---")
print(f"  Recency  — médiane : {rfm['recency'].median():.0f} jours")
print(f"  Frequency— médiane : {rfm['frequency'].median():.0f} commandes")
print(f"  Monetary — médiane : {rfm['monetary'].median():.2f} BRL")
print(f"  Monetary — moyenne : {rfm['monetary'].mean():.2f} BRL")
print(f"\n  Écart moyenne/médiane monetary : "
      f"{(rfm['monetary'].mean()-rfm['monetary'].median())/rfm['monetary'].median()*100:.1f}%")
print(f"  → Distribution skewed confirmée — médiane représente mieux")


   ÉTAPE 7 — RFM : AGRÉGATION PAR CLIENT
   Cours 3 §3 — tendance centrale par client
   EDA : 96.9% mono-achat → Recency critique

✓ Table RFM : (93350, 17)

--- Stats RFM ---
  Recency  — médiane : 268 jours
  Frequency— médiane : 1 commandes
  Monetary — médiane : 107.78 BRL
  Monetary — moyenne : 165.17 BRL

  Écart moyenne/médiane monetary : 53.2%
  → Distribution skewed confirmée — médiane représente mieux


In [8]:
# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 8 — SCORING RFM")
print("   Transformer R, F, M en scores 1-5")
print("="*60 + "\n")
# -------------------------------------------------------

# Score Recency : plus c'est récent = meilleur score
# Donc on inverse : recency faible → score 5
rfm['r_score'] = pd.qcut(rfm['recency'], q=5,
                          labels=[5,4,3,2,1])

# Score Frequency : plus c'est fréquent = meilleur score
# qcut ne marche pas si trop peu de valeurs uniques
# 96.9% ont frequency=1 → on gère ce cas
try:
    rfm['f_score'] = pd.qcut(rfm['frequency'], q=5,
                              labels=[1,2,3,4,5],
                              duplicates='drop')
except Exception:
    rfm['f_score'] = pd.cut(rfm['frequency'],
                             bins=[0,1,2,3,5,rfm['frequency'].max()+1],
                             labels=[1,2,3,4,5])

# Score Monetary : plus c'est élevé = meilleur score
rfm['m_score'] = pd.qcut(rfm['monetary'], q=5,
                          labels=[1,2,3,4,5],
                          duplicates='drop')

# Score RFM global
rfm['r_score'] = rfm['r_score'].astype(int)
rfm['f_score'] = rfm['f_score'].astype(int)
rfm['m_score'] = rfm['m_score'].astype(int)
rfm['rfm_score'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']

print(f"✓ Scores RFM calculés")
print(f"\n--- Distribution rfm_score ---")
print(rfm['rfm_score'].describe())


   ÉTAPE 8 — SCORING RFM
   Transformer R, F, M en scores 1-5

✓ Scores RFM calculés

--- Distribution rfm_score ---
count    93350.000000
mean         7.034804
std          2.053484
min          3.000000
25%          6.000000
50%          7.000000
75%          8.000000
max         15.000000
Name: rfm_score, dtype: float64


In [9]:
# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 9 — SEGMENTATION CLIENT")
print("   Transformer le score RFM en segments métier")
print("="*60 + "\n")
# -------------------------------------------------------

def segmenter_client(row):
    """
    Segmentation basée sur les scores R, F, M
    Logique métier — pas algorithmique
    """
    r, f, m = row['r_score'], row['f_score'], row['m_score']

    if r >= 4 and f >= 4 and m >= 4:
        return 'VIP'
    elif r >= 3 and f >= 3:
        return 'Fidèle'
    elif r >= 4 and f <= 2:
        return 'Nouveau'
    elif r <= 2 and f >= 3:
        return 'À risque'
    elif r <= 2 and f <= 2:
        return 'Perdu'
    else:
        return 'Intermédiaire'

rfm['segment'] = rfm.apply(segmenter_client, axis=1)

print(f"✓ Segmentation calculée")
print(f"\n--- Distribution des segments ---")
seg_dist = rfm['segment'].value_counts()
for seg, count in seg_dist.items():
    pct = count / len(rfm) * 100
    print(f"  {seg:<15} : {count:>7,} ({pct:.1f}%)")


   ÉTAPE 9 — SEGMENTATION CLIENT
   Transformer le score RFM en segments métier

✓ Segmentation calculée

--- Distribution des segments ---
  Nouveau         :  37,272 (39.9%)
  Perdu           :  37,196 (39.8%)
  Intermédiaire   :  18,654 (20.0%)
  Fidèle          :     124 (0.1%)
  À risque        :      71 (0.1%)
  VIP             :      33 (0.0%)


In [10]:
# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 10 — TARGET ML : is_churned")
print("   Variable cible pour le modèle de churn")
print("   EDA : 96.9% mono-achat → définition du churn")
print("="*60 + "\n")
# -------------------------------------------------------

# Définition du churn :
# Un client est "churned" s'il n'a pas commandé
# depuis plus de 180 jours ET n'a commandé qu'une fois
# Justification : médiane recency + analyse mono-achat EDA

SEUIL_CHURN_JOURS = 180

rfm['is_churned'] = (
    (rfm['recency'] > SEUIL_CHURN_JOURS) &
    (rfm['frequency'] == 1)
).astype(int)

n_churned = rfm['is_churned'].sum()
pct_churned = n_churned / len(rfm) * 100

print(f"✓ Seuil churn : {SEUIL_CHURN_JOURS} jours")
print(f"✓ Clients churned  : {n_churned:,} ({pct_churned:.1f}%)")
print(f"✓ Clients actifs   : {len(rfm)-n_churned:,} "
      f"({100-pct_churned:.1f}%)")
print(f"\n→ is_churned = TARGET du modèle ML dans 06_machine_learning")


   ÉTAPE 10 — TARGET ML : is_churned
   Variable cible pour le modèle de churn
   EDA : 96.9% mono-achat → définition du churn

✓ Seuil churn : 180 jours
✓ Clients churned  : 64,209 (68.8%)
✓ Clients actifs   : 29,141 (31.2%)

→ is_churned = TARGET du modèle ML dans 06_machine_learning


In [11]:
# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 11 — ONE-HOT ENCODING")
print("   Cours 1 P4 — préparer les variables nominales pour ML")
print("   Cours 3 §2 — nominale → pas de calcul numérique direct")
print("="*60 + "\n")
# -------------------------------------------------------

# customer_state → One-Hot Encoding
# Cours 1 P4 : drop_first=True pour éviter multicolinéarité
rfm_encoded = pd.get_dummies(
    rfm,
    columns=['customer_state'],
    prefix='state',
    drop_first=True  # Cours 1 P4 — Pro tip : éviter multicolinéarité
)

print(f"✓ Colonnes avant encoding : {rfm.shape[1]}")
print(f"✓ Colonnes après encoding : {rfm_encoded.shape[1]}")
print(f"✓ Colonnes state créées   : "
      f"{len([c for c in rfm_encoded.columns if c.startswith('state_')])}")


   ÉTAPE 11 — ONE-HOT ENCODING
   Cours 1 P4 — préparer les variables nominales pour ML
   Cours 3 §2 — nominale → pas de calcul numérique direct

✓ Colonnes avant encoding : 23
✓ Colonnes après encoding : 48
✓ Colonnes state créées   : 26


In [12]:
# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 12 — RAPPORT FINAL + EXPORT")
print("="*60 + "\n")
# -------------------------------------------------------

import os
os.makedirs('../data/processed', exist_ok=True)

# Rapport final
print(f"{'='*50}")
print(f"  TABLE FEATURES FINALE")
print(f"{'='*50}")
print(f"  Lignes (clients uniques) : {len(rfm):>10,}")
print(f"  Colonnes features        : {len(rfm_encoded.columns):>10}")
print(f"  Target is_churned        : {rfm['is_churned'].sum():>10,} "
      f"({rfm['is_churned'].mean()*100:.1f}%)")
print(f"\n  Nulls restants :")
nulls = rfm_encoded.isnull().sum()
print(nulls[nulls > 0] if len(nulls[nulls > 0]) > 0
      else "  ✓ Aucun null")

# Export table RFM avec segments (pour Power BI)
rfm.to_csv('../data/processed/olist_rfm.csv', index=False)
print(f"\n✓ olist_rfm.csv → data/processed/")

# Export table features encodées (pour ML)
rfm_encoded.to_csv('../data/processed/olist_features.csv', index=False)
print(f"✓ olist_features.csv → data/processed/")

# Export pour Power BI
os.makedirs('../data/exports', exist_ok=True)
rfm[['customer_unique_id','recency','frequency',
     'monetary','rfm_score','segment',
     'panier_moyen','delai_moyen','score_moyen',
     'customer_state','is_churned']].to_csv(
    '../data/exports/rfm_dashboard.csv', index=False)
print(f"✓ rfm_dashboard.csv → data/exports/ (Power BI)")

print("\n" + "="*60)
print("   ✓ MILESTONE 3 — FEATURE ENGINEERING TERMINÉ")
print("   → 2 fichiers créés :")
print("     olist_rfm.csv      — segments + RFM (Power BI)")
print("     olist_features.csv — features ML encodées")
print("   → Prochain : 06_machine_learning.ipynb")
print("="*60)


   ÉTAPE 12 — RAPPORT FINAL + EXPORT

  TABLE FEATURES FINALE
  Lignes (clients uniques) :     93,350
  Colonnes features        :         48
  Target is_churned        :     64,209 (68.8%)

  Nulls restants :
score_moyen        603
is_insatisfait     603
pct_credit           1
mensualites_moy      1
dtype: int64

✓ olist_rfm.csv → data/processed/
✓ olist_features.csv → data/processed/
✓ rfm_dashboard.csv → data/exports/ (Power BI)

   ✓ MILESTONE 3 — FEATURE ENGINEERING TERMINÉ
   → 2 fichiers créés :
     olist_rfm.csv      — segments + RFM (Power BI)
     olist_features.csv — features ML encodées
   → Prochain : 06_machine_learning.ipynb


In [13]:
# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 11b — TRAITEMENT NULLS RÉSIDUELS")
print("   Cours 3 §3 — imputation médiane avant ML")
print("="*60 + "\n")
# -------------------------------------------------------

# score_moyen et is_insatisfait : 603 clients sans aucun avis
# Ces clients n'ont jamais laissé de review
# Imputer score_moyen avec la médiane globale
# Imputer is_insatisfait avec 0 (pas d'insatisfaction connue)
mediane_score = rfm['score_moyen'].median()
rfm['score_moyen']     = rfm['score_moyen'].fillna(mediane_score)
rfm['is_insatisfait']  = rfm['is_insatisfait'].fillna(0).astype(int)

# pct_credit et mensualites_moy : 1 client sans paiement enregistré
# Imputer avec la médiane
rfm['pct_credit']       = rfm['pct_credit'].fillna(rfm['pct_credit'].median())
rfm['mensualites_moy']  = rfm['mensualites_moy'].fillna(
                           rfm['mensualites_moy'].median())

print(f"✓ score_moyen    imputé avec médiane = {mediane_score:.2f}")
print(f"✓ is_insatisfait imputé avec 0")
print(f"✓ pct_credit     imputé avec médiane")
print(f"✓ mensualites_moy imputé avec médiane")
print(f"\n✓ Nulls restants : {rfm.isnull().sum().sum()}")


   ÉTAPE 11b — TRAITEMENT NULLS RÉSIDUELS
   Cours 3 §3 — imputation médiane avant ML

✓ score_moyen    imputé avec médiane = 5.00
✓ is_insatisfait imputé avec 0
✓ pct_credit     imputé avec médiane
✓ mensualites_moy imputé avec médiane

✓ Nulls restants : 0


In [2]:
# ============================================================
# EXPORT POWER BI — CA par catégorie
# Coller dans 05_feature_engineering.ipynb
# OU créer un nouveau notebook 07_exports_powerbi.ipynb
# ============================================================

import sys
import os
sys.path.append('..')
from src.utils import *

os.makedirs('../data/exports', exist_ok=True)

# -------------------------------------------------------
print("\n" + "="*60)
print("   CHARGEMENT — tables nécessaires")
print("="*60 + "\n")
# -------------------------------------------------------

# Recharger les tables depuis processed/
# Chaque notebook est autonome — pas de dépendance mémoire
items    = pd.read_csv('../data/processed/items_clean.csv')
products = pd.read_csv('../data/processed/products_clean.csv')
orders   = pd.read_csv('../data/processed/orders_clean.csv',
                        parse_dates=['order_purchase_timestamp'])
customers= pd.read_csv('../data/processed/customers_clean.csv')

print(f"✓ items    : {items.shape}")
print(f"✓ products : {products.shape}")
print(f"✓ orders   : {orders.shape}")
print(f"✓ customers: {customers.shape}")


# -------------------------------------------------------
print("\n" + "="*60)
print("   EXPORT 1 — CA par catégorie (Bar Chart + Pareto)")
print("="*60 + "\n")
# -------------------------------------------------------

# Jointure items + products
items_prod = items.merge(
    products[['product_id',
              'product_category_name_english']],
    on='product_id',
    how='left'
)

# CA = price + freight_value
items_prod['revenue'] = items_prod['price'] + items_prod['freight_value']

# Agrégation par catégorie
ca_par_categorie = (items_prod
    .groupby('product_category_name_english')
    .agg(
        ca_total    = ('revenue',   'sum'),
        nb_ventes   = ('order_id',  'count'),
        prix_moyen  = ('price',     'mean'),
        prix_median = ('price',     'median')
    )
    .round(2)
    .sort_values('ca_total', ascending=False)
    .reset_index()
)

# Rang + % CA cumulé pour le Pareto
ca_par_categorie['rang']         = range(1, len(ca_par_categorie)+1)
ca_par_categorie['ca_pct']       = (
    ca_par_categorie['ca_total'] /
    ca_par_categorie['ca_total'].sum() * 100
).round(2)
ca_par_categorie['ca_cumul_pct'] = (
    ca_par_categorie['ca_pct'].cumsum().round(2)
)

ca_par_categorie.to_csv('../data/exports/ca_par_categorie.csv', index=False)
print(f"✓ ca_par_categorie.csv exporté — {len(ca_par_categorie)} catégories")
print(f"\n--- Top 10 ---")
print(ca_par_categorie[[
    'product_category_name_english',
    'ca_total', 'nb_ventes',
    'rang', 'ca_cumul_pct'
]].head(10).to_string(index=False))


# -------------------------------------------------------
print("\n" + "="*60)
print("   EXPORT 2 — CA mensuel (Line Chart)")
print("="*60 + "\n")
# -------------------------------------------------------

orders_items = orders.merge(
    items[['order_id','price','freight_value']],
    on='order_id', how='left'
)
orders_items['revenue']    = orders_items['price'] + orders_items['freight_value']
orders_items['année_mois'] = (orders_items['order_purchase_timestamp']
                               .dt.strftime('%Y-%m'))

ca_mensuel = (orders_items
    .groupby('année_mois')
    .agg(
        ca_total      = ('revenue',   'sum'),
        nb_commandes  = ('order_id',  'nunique'),
        panier_moyen  = ('revenue',   'mean')
    )
    .round(2)
    .reset_index()
    .sort_values('année_mois')
)

ca_mensuel.to_csv('../data/exports/ca_mensuel.csv', index=False)
print(f"✓ ca_mensuel.csv exporté — {len(ca_mensuel)} mois")
print(ca_mensuel.to_string(index=False))


# -------------------------------------------------------
print("\n" + "="*60)
print("   EXPORT 3 — Commandes par état (Map)")
print("="*60 + "\n")
# -------------------------------------------------------

commandes_etat = (orders
    .merge(customers[['customer_id',
                       'customer_unique_id',
                       'customer_state']],
           on='customer_id', how='left')
    .groupby('customer_state')
    .agg(
        nb_commandes = ('order_id',          'count'),
        nb_clients   = ('customer_unique_id', 'nunique'),
        ca_total     = ('order_id',           'count')
    )
    .reset_index()
    .sort_values('nb_commandes', ascending=False)
)

# Ajouter les noms complets des états pour Power BI Map
noms_etats = {
    'SP':'São Paulo','RJ':'Rio de Janeiro','MG':'Minas Gerais',
    'RS':'Rio Grande do Sul','PR':'Paraná','SC':'Santa Catarina',
    'BA':'Bahia','DF':'Distrito Federal','ES':'Espírito Santo',
    'GO':'Goiás','PE':'Pernambuco','CE':'Ceará','MT':'Mato Grosso',
    'MS':'Mato Grosso do Sul','MA':'Maranhão','PA':'Pará',
    'AM':'Amazonas','RN':'Rio Grande do Norte','PI':'Piauí',
    'AL':'Alagoas','PB':'Paraíba','SE':'Sergipe','TO':'Tocantins',
    'RO':'Rondônia','AC':'Acre','AP':'Amapá','RR':'Roraima'
}
commandes_etat['estado_nome'] = (
    commandes_etat['customer_state'].map(noms_etats)
)

commandes_etat.to_csv('../data/exports/commandes_par_etat.csv', index=False)
print(f"✓ commandes_par_etat.csv exporté")
print(commandes_etat.head(10).to_string(index=False))


# -------------------------------------------------------
print("\n" + "="*60)
print("   EXPORT 4 — KPIs globaux (Cards)")
print("="*60 + "\n")
# -------------------------------------------------------

rfm = pd.read_csv('../data/processed/olist_rfm.csv')

kpis = {
    'ca_total':           round(orders_items['revenue'].sum(), 2),
    'nb_commandes':       orders['order_id'].nunique(),
    'nb_clients':         customers['customer_unique_id'].nunique(),
    'panier_moyen':       round(orders_items.groupby('order_id')
                               ['revenue'].sum().mean(), 2),
    'panier_median':      round(orders_items.groupby('order_id')
                               ['revenue'].sum().median(), 2),
    'taux_churn_pct':     round(rfm['is_churned'].mean() * 100, 1),
    'taux_mono_achat_pct':round((rfm['frequency']==1).mean() * 100, 1),
    'recency_mediane':    round(rfm['recency'].median(), 0),
    'score_satisfaction': round(rfm['score_moyen'].mean(), 2),
    'nb_categories':      ca_par_categorie['product_category_name_english'].nunique(),
}

kpis_df = pd.DataFrame(list(kpis.items()),
                        columns=['kpi', 'valeur'])
kpis_df.to_csv('../data/exports/kpis_globaux.csv', index=False)

print(f"✓ kpis_globaux.csv exporté")
print(f"\n{'KPI':<30} {'Valeur':>15}")
print("-" * 47)
for _, row in kpis_df.iterrows():
    print(f"  {row['kpi']:<30} {row['valeur']:>15}")


# -------------------------------------------------------
print("\n" + "="*60)
print("   RÉSUMÉ — FICHIERS EXPORTÉS POUR POWER BI")
print("="*60 + "\n")
# -------------------------------------------------------

exports = [
    ('ca_par_categorie.csv',   'Bar Chart Top 10 catégories + Pareto'),
    ('ca_mensuel.csv',         'Line Chart évolution CA mensuel'),
    ('commandes_par_etat.csv', 'Map + Bar Chart par état'),
    ('kpis_globaux.csv',       'Cards KPIs principaux'),
    ('rfm_dashboard.csv',      'Segments + Churn + Clusters'),
]

print(f"{'Fichier':<30} {'Utilisation Power BI'}")
print("-" * 65)
for fichier, usage in exports:
    taille = os.path.getsize(f'../data/exports/{fichier}')
    print(f"  {fichier:<30} {usage}")

print(f"\n✓ {len(exports)} fichiers prêts dans data/exports/")
print(f"\n→ Dans Power BI :")
print(f"  Obtenir données → Texte/CSV → sélectionner chaque fichier")
print(f"  Pas besoin de jointures complexes — tout est précalculé")


   CHARGEMENT — tables nécessaires

✓ items    : (112650, 11)
✓ products : (32951, 10)
✓ orders   : (99433, 8)
✓ customers: (99441, 5)

   EXPORT 1 — CA par catégorie (Bar Chart + Pareto)

✓ ca_par_categorie.csv exporté — 72 catégories

--- Top 10 ---
product_category_name_english   ca_total  nb_ventes  rang  ca_cumul_pct
                health_beauty 1441248.07       9670     1          9.10
                watches_gifts 1305541.61       5991     2         17.34
               bed_bath_table 1241681.72      11115     3         25.18
               sports_leisure 1156656.48       8641     4         32.48
        computers_accessories 1059272.40       7827     5         39.17
              furniture_decor  902511.79       8334     6         44.87
                   housewares  778397.77       6964     7         49.78
                   cool_stuff  719329.95       3796     8         54.32
                         auto  685384.32       4235     9         58.65
                 garden_too